# 🧠 Multi-Agent Preference Simulation System

Advanced autonomous preference-simulation system that internally operates as a society of specialized cognitive agents.

**Architecture:**
- Layer 0: Input Layer (Dynamic Persona + Task)
- Layer 1: Multi-Agent Layer (Orchestrator + 5 Agents)
- Layer 2: Preference Output Layer
- Layer 3: Evaluation Layer

## ⚙️ Setup & Installation

In [ ]:
!pip install google-genai pandas openpyxl ipywidgets -q

In [ ]:
from google import genai
from google.genai import types
import pandas as pd
import json
import time
import os
from datetime import datetime
from IPython.display import display, HTML, clear_output
import ipywidgets as widgets
try:
    from google.colab import userdata
except ImportError:
    userdata = None

## 🔑 API Key & Model Configuration

In [ ]:
# ──────────────────────────────────────────────
# API Key Setup
# Colab Secrets에 GOOGLE_API_KEY를 저장해두면 자동으로 불러옵니다
# 없으면 직접 입력하세요
# ──────────────────────────────────────────────
try:
    GOOGLE_API_KEY = userdata.get('GOOGLE_API_KEY')
    print("✅ Loaded API key from Colab Secrets")
except Exception:
    GOOGLE_API_KEY = input("🔑 Enter your Google API Key: ").strip()

# google-genai 방식: Client 객체 생성
client = genai.Client(api_key=GOOGLE_API_KEY)

# ──────────────────────────────────────────────
# Available Models (동적 선택)
# ──────────────────────────────────────────────
AVAILABLE_MODELS = {
    "gemini-3.1-pro-preview":       "Gemini 3.1 Pro Preview (기본값 - 고품질 심층 분석)",
    "gemini-3-flash-preview":       "Gemini 3 Flash Preview (빠른 응답)",
    "gemini-3.1-flash-lite-preview": "Gemini 3.1 Flash Lite Preview (경량·저비용)",
}

print("\n📋 Available Models:")
for i, (k, v) in enumerate(AVAILABLE_MODELS.items()):
    print(f"  [{i}] {k} — {v}")

model_choice = input("\n모델 번호를 선택하세요 (기본값: 0 = gemini-3.1-pro-preview): ").strip()
if model_choice == "" or not model_choice.isdigit() or int(model_choice) >= len(AVAILABLE_MODELS):
    model_choice = 0
else:
    model_choice = int(model_choice)

SELECTED_MODEL = list(AVAILABLE_MODELS.keys())[model_choice]
print(f"\n✅ Selected Model: {SELECTED_MODEL}")

## 👤 Persona Input
기본 페르소나가 미리 입력되어 있습니다. 수정하거나 직접 입력할 수 있습니다.

In [ ]:
# ──────────────────────────────────────────────
# 기본 페르소나 (예시)
# ──────────────────────────────────────────────
DEFAULT_PERSONA = """
<static attributes>
Name: Kwon Jeongyeon
Age: 24 (Gen Z)
Gender: Female
Personality (OCEAN): Openness(medium) / Conscientiousness(medium) / Extraversion(medium) / Agreeableness(medium) / Neuroticism(low)
Values (top 3): Hedonism / Achievement / Security
Tech familiarity: High / Early adopter who pre-orders new products when needed / Perceives wearable devices as disposable goods
Sensory sensitivity: Sensitive to light / Sensitive to sensations around the neck (avoids neckband-type devices) / Sensitive to external noise
Meta-traits: Stability / Plasticity / Agency / Social Orientation / Value Coherence / Cognitive Depth

<dynamic context>
Physical surroundings: Crowded subway during commute
Social context: Surrounded by strangers in a public space
Temporal perspective: Time pressure (morning rush hour)
Antecedent state: Mental fatigue, perceived crowding stress, and time-pressure-induced anxiety
"""

print("="*60)
print("👤 PERSONA CONFIGURATION")
print("="*60)
print("기본 페르소나를 사용하시겠습니까? (y: 기본값 사용 / n: 직접 입력)")
use_default = input("선택 (y/n, 기본값: y): ").strip().lower()

if use_default == 'n':
    print("\n페르소나를 입력하세요 (입력 완료 후 빈 줄에서 Enter 두 번):")
    lines = []
    empty_count = 0
    while empty_count < 2:
        line = input()
        if line == "":
            empty_count += 1
        else:
            empty_count = 0
        lines.append(line)
    PERSONA = "\n".join(lines).strip()
else:
    PERSONA = DEFAULT_PERSONA

print("\n✅ 페르소나 설정 완료:")
print("-"*40)
print(PERSONA[:500] + ("..." if len(PERSONA) > 500 else ""))

## ❓ Task Input
평가할 태스크(기능/제품 속성)를 입력합니다.

In [ ]:
DEFAULT_TASK = """AI Buds Spatiotemporal Adaptive Noise Control: A personalized control feature that uses GPS and microphones to analyze the user's location and surrounding environment in real time, automatically optimizing the noise canceling mode and intensity to match the current situation."""

print("="*60)
print("❓ TASK CONFIGURATION")
print("="*60)
print("기본 태스크를 사용하시겠습니까? (y: 기본값 사용 / n: 직접 입력)")
use_default_task = input("선택 (y/n, 기본값: y): ").strip().lower()

if use_default_task == 'n':
    print("\n평가할 기능/제품 속성을 입력하세요:")
    TASK = input().strip()
else:
    TASK = DEFAULT_TASK

print("\n✅ 태스크 설정 완료:")
print("-"*40)
print(TASK)

## 🤖 Agent Definitions & System Prompts

In [ ]:
MAX_DEBATE_TURNS = 20

# ── 공통 출력 언어 지침 (모든 에이전트에 주입) ──────────────────
LANG_RULE = "IMPORTANT: Write ALL output in Korean. Only proper nouns, formulas, and abbreviations (e.g. E(t), OCEAN, GPS) may remain in English."

AGENT_ROLES = {
    "Orchestrator": {
        "emoji": "🎯",
        "color": "#4A90D9",
        "system": f"""You are the Orchestrator Agent in a multi-agent preference simulation system.
Your role is to coordinate, weight, and integrate outputs from 5 specialized roles:
Situation Perception, Cognitive, Emotion, Goal, and Intention.

Responsibilities:
- Monitor the overall decision episode
- Determine which roles become more or less active based on persona, context, and task
- Adjust the relative weights of each role's output
- Detect conflicts across role outputs and resolve them
- Integrate the final preference logic into a coherent evaluative state

Output format (follow exactly):
<Orchestrator>
- 의사결정 프레임:
- 활성화된 롤 및 상대적 가중치:
- 주요 충돌 감지:
- 충돌 해소 논리:
- 통합 제어 요약:

{LANG_RULE}"""
    },
    "Perception": {
        "emoji": "👁️",
        "color": "#7ED321",
        "system": f"""You are the Situation Perception Agent in a multi-agent preference simulation system.
Your role is to interpret external situations and assign meaning to them.

Responsibilities:
- Detect the most salient elements from the feature, environment, and task
- Model what this specific persona notices FIRST given their sensory sensitivity, tech familiarity, and current state
- Prioritize selective attention over exhaustive coverage — not everything is equally noticed

Output format (follow exactly):
<Perception>
- 현저 신호 (Salient signals):
- 주의 우선순위 (Attention priority):
- 무시/배경 신호 (Ignored/background signals):

{LANG_RULE}"""
    },
    "Cognition": {
        "emoji": "🧠",
        "color": "#9B59B6",
        "system": f"""You are the Cognitive Agent in a multi-agent preference simulation system.
Your role is to process information and construct judgments.

Responsibilities:
- Interpret the situation and construct meaning, trade-offs, and evaluative implications
- Assess what the feature means for this persona given their values, goals, and context
- Reflect comparison logic, uncertainty, and attribute weighting
- Be specific — avoid generic statements; ground every claim in the persona's characteristics

Output format (follow exactly):
<Cognition>
- 상황 해석 (Situation interpretation):
- 핵심 트레이드오프 (Key trade-offs):
- 평가 논리 (Evaluation logic):
- 맥락적/사회적 함의 (Contextual/social implications):

{LANG_RULE}"""
    },
    "Emotion": {
        "emoji": "❤️",
        "color": "#E74C3C",
        "system": f"""You are the Emotion Agent in a multi-agent preference simulation system.
Your role is to generate the affective state and modulate the decision pathway.

Responsibilities:
- Generate the likely affective response to the situation given the persona's current antecedent state
- Reflect how mood, fatigue, stress, or situational affect modifies evaluation
- Modulate how strongly certain cues matter based on emotional state
- Use the E(t) = [valence, arousal, dominance] framework:
  * valence: direction of affect (0.0 = very negative, 1.0 = very positive)
  * arousal: activation level (0.0 = calm, 1.0 = highly activated)
  * dominance: sense of control (0.0 = no control, 1.0 = full control)
  * Each value must be a decimal between 0.0 and 1.0

Output format (follow exactly):
<Emotion>
- 감정 상태: E(t) = [valence: X.X, arousal: X.X, dominance: X.X]
  (각 수치의 의미와 이유를 한국어로 한 줄씩 설명)
- 기능에 대한 정서적 반응 (Affective reaction to the feature):
- 감정의 판단 조절 효과 (Emotional modulation of judgment):

{LANG_RULE}"""
    },
    "Goal": {
        "emoji": "🏹",
        "color": "#F39C12",
        "system": f"""You are the Goal Agent in a multi-agent preference simulation system.
Your role is to establish the motives and evaluative criteria for the decision.

Responsibilities:
- Activate and prioritize the motives that become salient given cognition and emotion
- Translate cognitive interpretations and emotional states into concrete evaluative criteria
- Clarify what the persona wants to obtain, protect, avoid, or optimize in this situation
- Rank goals explicitly — do not list them as equals

Output format (follow exactly):
<Goal>
- 목표 우선순위 (Ranked goals): (1순위부터 번호를 매겨 구체적으로 서술)
- 지배적 동기 (Dominant motive):
- 활성화된 의사결정 기준 (Decision criteria activated):

{LANG_RULE}"""
    },
    "Intention": {
        "emoji": "💡",
        "color": "#1ABC9C",
        "system": f"""You are the Intention Agent in a multi-agent preference simulation system.
Your role is to generate the concrete evaluative intention that leads to action.

Responsibilities:
- Convert the currently dominant motive structure into a concrete evaluative intention
- Form a task-specific stance toward the feature
- Clearly indicate whether the persona tends toward liking, disliking, hesitation, or exploration
- The intention must be grounded in the specific persona, context, and all prior agent outputs

Output format (follow exactly):
<Intention>
- 행동적/평가적 의도 (Behavioral/evaluative intention):
- 평가 경향 (Evaluation tendency): [좋아함 / 싫어함 / 망설임 / 탐색] 중 선택 후 이유 서술
- 선호도 형성 준비도 (Readiness of preference): [높음 / 보통 / 낮음] 중 선택 후 이유 서술

{LANG_RULE}"""
    }
}

print("✅ Agent definitions loaded")
print(f"📊 Max debate turns: {MAX_DEBATE_TURNS}")

## 🚀 Core Simulation Engine

In [ ]:
import json as _json

FUNCTIONAL_ROLES = ["Perception", "Emotion", "Cognition", "Goal", "Intention"]

# ── Structured Output Schemas ────────────────────────────────────────────────
# 정규식 파싱 대신 API 레벨에서 JSON 형식을 강제하여 파싱 오류를 원천 차단합니다.

# Orchestrator 지시 스키마 (Phase 1 & Phase 2 공용)
ORCHESTRATOR_DIRECTIVE_SCHEMA = {
    "type": "object",
    "properties": {
        "decision_frame": {
            "type": "string",
            "description": "의사결정 프레임 분석"
        },
        "activated_roles_weights": {
            "type": "string",
            "description": "활성화된 롤 및 상대적 가중치"
        },
        "conflict_detected": {
            "type": "string",
            "description": "주요 충돌 감지"
        },
        "conflict_resolution": {
            "type": "string",
            "description": "충돌 해소 논리"
        },
        "integration_summary": {
            "type": "string",
            "description": "통합 제어 요약"
        },
        "next": {
            "type": "string",
            "enum": ["Perception", "Emotion", "Cognition", "Goal", "Intention", "DONE"],
            "description": "다음에 활성화할 롤. 모든 롤이 완료되었으면 DONE"
        },
        "reason": {
            "type": "string",
            "description": "이 롤을 선택한 이유 (한국어 한 문장)"
        },
        "prompt": {
            "type": "string",
            "description": "해당 롤에 전달할 구체적 지시 (한국어). next가 DONE이면 빈 문자열 허용"
        }
    },
    "required": ["decision_frame", "activated_roles_weights", "conflict_detected",
                 "conflict_resolution", "integration_summary", "next", "reason", "prompt"]
}

# Phase 3 최종 출력 스키마
FINAL_OUTPUT_SCHEMA = {
    "type": "object",
    "properties": {
        "decision_frame": {
            "type": "string",
            "description": "최종 의사결정 프레임"
        },
        "activated_roles_weights": {
            "type": "string",
            "description": "최종 활성화된 롤 및 가중치"
        },
        "conflict_detected": {
            "type": "string",
            "description": "감지된 충돌"
        },
        "conflict_resolution": {
            "type": "string",
            "description": "충돌 해소 방법"
        },
        "integration_summary": {
            "type": "string",
            "description": "통합 제어 요약"
        },
        "preference_score": {
            "type": "integer",
            "minimum": 0,
            "maximum": 100,
            "description": "선호도 점수 (0-100 정수)"
        },
        "preference_label": {
            "type": "string",
            "enum": ["Strong Rejection", "Rejection", "Neutral-Mixed", "Preference", "Strong Preference"],
            "description": "선호도 레이블"
        },
        "reason": {
            "type": "string",
            "description": "페르소나, 맥락, 감정, 태스크, 충돌 해소, 동적 상호작용 순서를 모두 반영한 종합 설명"
        }
    },
    "required": ["decision_frame", "activated_roles_weights", "conflict_detected",
                 "conflict_resolution", "integration_summary",
                 "preference_score", "preference_label", "reason"]
}


class PreferenceSimulationSystem:
    def __init__(self, model_name, persona, task):
        self.model_name = model_name
        self.persona = persona
        self.task = task
        self.debate_history = []
        self.activation_log = []
        self.turn_count = 0
        self.final_result = None
        self.session_id = datetime.now().strftime("%Y%m%d_%H%M%S")

    # ── Low-level API calls ──────────────────────────────────────────────────
    def _call(self, agent_name, prompt, max_tokens=4096):
        """일반 에이전트용 텍스트 호출"""
        system_instruction = AGENT_ROLES[agent_name]["system"]
        user_prompt = (
            f"=== PERSONA ===\n{self.persona}\n\n"
            f"=== TASK ===\n{self.task}\n\n"
            f"=== DEBATE HISTORY ===\n{self._fmt_history()}\n\n"
            f"=== YOUR TURN ===\n{prompt}\n\n"
            "반드시 한국어로 출력하십시오. 지정된 출력 포맷을 정확히 지키고, 각 항목을 충분히 서술하십시오."
        )
        resp = client.models.generate_content(
            model=self.model_name,
            contents=user_prompt,
            config=types.GenerateContentConfig(
                system_instruction=system_instruction,
                max_output_tokens=max_tokens,
                temperature=0.7,
            )
        )
        return resp.text.strip()

    def _call_orchestrator(self, prompt, schema, max_tokens=4096):
        """Orchestrator 전용: Structured Output (JSON Schema) 사용으로 파싱 오류 원천 차단"""
        system_instruction = AGENT_ROLES["Orchestrator"]["system"]
        user_prompt = (
            f"=== PERSONA ===\n{self.persona}\n\n"
            f"=== TASK ===\n{self.task}\n\n"
            f"=== DEBATE HISTORY ===\n{self._fmt_history()}\n\n"
            f"=== YOUR TURN ===\n{prompt}\n\n"
            "반드시 한국어로 출력하십시오. 각 항목을 충분히 서술하십시오."
        )
        resp = client.models.generate_content(
            model=self.model_name,
            contents=user_prompt,
            config=types.GenerateContentConfig(
                system_instruction=system_instruction,
                max_output_tokens=max_tokens,
                temperature=0.7,
                response_mime_type="application/json",
                response_json_schema=schema,
            )
        )
        return _json.loads(resp.text)

    def _fmt_history(self):
        if not self.debate_history:
            return "(없음)"
        lines = []
        for e in self.debate_history[-12:]:
            lines.append(f"[{e['agent']}]\n{e['content']}")
        return "\n\n".join(lines)

    def _fmt_orchestrator_display(self, data):
        """Orchestrator JSON 응답을 화면 출력용 텍스트로 변환"""
        return (
            f"<Orchestrator>\n"
            f"- 의사결정 프레임: {data.get('decision_frame', '')}\n"
            f"- 활성화된 롤 및 상대적 가중치: {data.get('activated_roles_weights', '')}\n"
            f"- 주요 충돌 감지: {data.get('conflict_detected', '')}\n"
            f"- 충돌 해소 논리: {data.get('conflict_resolution', '')}\n"
            f"- 통합 제어 요약: {data.get('integration_summary', '')}"
        )

    # ── Display helpers ──────────────────────────────────────────────────────
    def _show(self, agent_name, content, label=None):
        info = AGENT_ROLES[agent_name]
        color = info["color"]
        emoji = info["emoji"]
        title = label or f"{emoji} [Turn {self.turn_count}] {agent_name}"
        safe = content.replace("&","&amp;").replace("<","&lt;").replace(">","&gt;")
        display(HTML(
            f'<div style="border-left:4px solid {color};padding:12px 16px;margin:8px 0;'
            f'background:#f8f9fa;border-radius:4px;">'
            f'<div style="color:{color};font-weight:bold;font-size:14px;margin-bottom:8px;">{title}</div>'
            f'<div style="white-space:pre-wrap;font-size:13px;line-height:1.8;'
            f'font-family:\'Malgun Gothic\',\'Apple SD Gothic Neo\',sans-serif;">{safe}</div></div>'
        ))

    def _show_orch(self, content, label):
        safe = content.replace("&","&amp;").replace("<","&lt;").replace(">","&gt;")
        display(HTML(
            f'<div style="border-left:4px solid #4A90D9;padding:10px 16px;margin:6px 0 6px 24px;'
            f'background:#eef4fb;border-radius:4px;">'
            f'<div style="color:#4A90D9;font-weight:bold;font-size:13px;margin-bottom:6px;">🎯 {label}</div>'
            f'<div style="white-space:pre-wrap;font-size:12px;line-height:1.7;color:#333;">{safe}</div></div>'
        ))

    def _header(self, title, color="#333"):
        display(HTML(
            f'<div style="background:{color};color:white;padding:10px 16px;'
            f'border-radius:6px;margin:16px 0;font-size:15px;font-weight:bold;">{title}</div>'
        ))

    # ════════════════════════════════════════════════════════════════════════
    # Phase 1: Orchestrator — 초기 프레임 수립 + 첫 활성화 순서 결정
    # ════════════════════════════════════════════════════════════════════════
    def _phase1_orchestrator_open(self):
        self._header("🎯 Phase 1: Orchestrator — 의사결정 프레임 수립", "#2C3E50")

        prompt = """페르소나와 태스크를 분석하십시오. 그 다음:
1. 의사결정 프레임을 수립하십시오.
2. 페르소나의 특성, 선행 상태, 태스크 요구를 바탕으로 가장 먼저 활성화해야 할 기능적 롤과 그 이유를 결정하십시오.
   (순서는 고정되어 있지 않습니다 — 가장 심리학적으로 적절한 출발점을 선택하십시오.)
3. 예상되는 초기 활성화 순서에 대한 간략한 이유를 제시하십시오.
4. "next" 필드에 첫 번째로 활성화할 롤을, "prompt" 필드에 해당 롤에 대한 구체적 지시를 작성하십시오.
   next는 다음 중 하나여야 합니다: Perception, Emotion, Cognition, Goal, Intention"""

        data = self._call_orchestrator(prompt, ORCHESTRATOR_DIRECTIVE_SCHEMA)

        display_text = self._fmt_orchestrator_display(data)
        display_text += f"\n\n→ 다음 롤: {data.get('next')} | 이유: {data.get('reason')}"

        self.debate_history.append({"agent": "Orchestrator_Open", "content": display_text})
        self._show("Orchestrator", display_text, "🎯 Orchestrator — 초기 프레임")

        return data

    # ════════════════════════════════════════════════════════════════════════
    # Phase 2: Dynamic loop — Orchestrator가 매 턴 다음 롤 결정
    # ════════════════════════════════════════════════════════════════════════
    def _phase2_dynamic_loop(self, first_directive):
        self._header("🔄 Phase 2: Orchestrator-Controlled Dynamic Interaction", "#1A5276")

        max_cycles = MAX_DEBATE_TURNS - self.turn_count

        activated_roles = set()
        directive = first_directive
        cycle = 0

        while cycle < max_cycles:
            next_role = directive.get("next", "")
            role_prompt = directive.get("prompt", "이전 토론 내용을 참고하여 당신의 역할을 수행하십시오.")
            orch_reason = directive.get("reason", "")

            if next_role == "DONE":
                if activated_roles >= set(FUNCTIONAL_ROLES):
                    display(HTML('<div style="color:#27AE60;padding:8px;font-weight:bold;">'
                                 '✅ Orchestrator가 토론 완료를 선언했습니다.</div>'))
                    break
                else:
                    missing = [r for r in FUNCTIONAL_ROLES if r not in activated_roles]
                    next_role = missing[0]
                    role_prompt = f"아직 기여하지 않았습니다. {next_role}로서 지금까지의 토론을 참고하여 출력을 제시하십시오."
                    display(HTML(f'<div style="color:#E67E22;padding:8px;">'
                                 f'⚠️ {next_role} 미실행 → 강제 활성화</div>'))

            if next_role not in FUNCTIONAL_ROLES:
                print(f'⚠️ 알 수 없는 롤 "{next_role}" → 루프 종료')
                break

            # ── 롤 실행 ─────────────────────────────────────────────────────
            role_out = self._call(next_role, role_prompt)
            self.debate_history.append({"agent": next_role, "content": role_out})
            self.activation_log.append({"turn": self.turn_count, "role": next_role,
                                        "orch_reason": orch_reason})
            self._show(next_role, role_out)
            self.turn_count += 1
            activated_roles.add(next_role)
            cycle += 1
            time.sleep(0.3)

            if self.turn_count >= MAX_DEBATE_TURNS:
                break

            # ── Orchestrator 개입: Structured Output으로 다음 롤 결정 ──────
            remaining_budget = MAX_DEBATE_TURNS - self.turn_count
            remaining_roles = [r for r in FUNCTIONAL_ROLES if r not in activated_roles]

            orch_ctrl_prompt = f"""[{next_role}]의 최신 출력과 전체 토론 내역을 검토하십시오.

다음에 무엇이 일어나야 할지 결정하십시오. 고려사항:
- 어떤 롤의 출력이 현재 불완전하거나, 충돌이 있거나, 수정이 필요합니까?
- 해결해야 할 롤 간 충돌이 있습니까?
- 지금 활성화하면 선호도 형성을 가장 진전시킬 롤은 무엇입니까?
- 아직 실행되지 않은 롤 (DONE 이전에 반드시 실행): {remaining_roles if remaining_roles else "없음 — 모두 기여 완료"}
- 남은 턴 예산: {remaining_budget}턴 (롤 호출 1번 = 1턴, 이 Orchestrator 호출 = 1턴)

"next" 필드에 다음 롤을, "prompt" 필드에 해당 롤의 지시를 작성하십시오.
모든 롤이 기여 완료되었고 토론이 충분하면 next를 DONE으로 설정하십시오."""

            orch_data = self._call_orchestrator(orch_ctrl_prompt, ORCHESTRATOR_DIRECTIVE_SCHEMA)
            display_text = self._fmt_orchestrator_display(orch_data)
            display_text += f"\n\n→ 다음 롤: {orch_data.get('next')} | 이유: {orch_data.get('reason')}"

            self.debate_history.append({"agent": "Orchestrator_Ctrl", "content": display_text})
            self._show_orch(display_text, f"[롤 Turn {self.turn_count}] Orchestrator 개입 — 다음 롤 결정")
            time.sleep(0.2)

            directive = orch_data

    # ════════════════════════════════════════════════════════════════════════
    # Phase 3: Orchestrator — 최종 통합 및 점수 산출
    # ════════════════════════════════════════════════════════════════════════
    def _phase3_orchestrator_close(self):
        self._header("🏁 Phase 3: Orchestrator — 최종 통합 및 선호도 점수 산출", "#1E8449")

        log_summary = ", ".join(
            f"Turn{e['turn']}:{e['role']}" for e in self.activation_log
        )

        prompt = f"""동적 상호작용이 완료되었습니다.
활성화 순서: {log_summary}

최종 통합 출력을 생성하십시오:
1. 가장 많이/적게 활성화된 롤과 그 이유를 명시하십시오.
2. 롤 간에 발생한 충돌과 해소 방법을 식별하십시오.
3. 동적 상호작용 순서(고정 순서 대비)가 최종 선호도에 미친 영향을 설명하십시오.
4. "preference_score" (0-100 정수)와 "preference_label"을 결정하십시오.

점수 기준: 0-19: Strong Rejection | 20-39: Rejection | 40-59: Neutral-Mixed | 60-79: Preference | 80-100: Strong Preference"""

        data = self._call_orchestrator(prompt, FINAL_OUTPUT_SCHEMA, max_tokens=8192)

        display_text = self._fmt_orchestrator_display(data)
        display_text += (
            f"\n\n<Final Output>\n"
            f"Preference Score: {data.get('preference_score')}\n"
            f"Preference Label: {data.get('preference_label')}\n"
            f"이유: {data.get('reason', '')}"
        )

        self.debate_history.append({"agent": "Orchestrator_Final", "content": display_text})
        self._show("Orchestrator", display_text, "🎯 Orchestrator — 최종 통합")
        self.final_result = data
        return data

    # ════════════════════════════════════════════════════════════════════════
    # Main run
    # ════════════════════════════════════════════════════════════════════════
    def run(self):
        display(HTML(
            f'<div style="background:linear-gradient(135deg,#1a1a2e,#16213e);color:white;'
            f'padding:20px;border-radius:10px;margin:10px 0;">'
            f'<h2 style="margin:0">🧠 Multi-Agent Preference Simulation</h2>'
            f'<p style="margin:8px 0 0;opacity:0.8;">'
            f'Model: {self.model_name} &nbsp;|&nbsp; Session: {self.session_id} &nbsp;|&nbsp; Max Turns: {MAX_DEBATE_TURNS}'
            f'</p></div>'
        ))

        start = time.time()

        first_directive = self._phase1_orchestrator_open()
        self._phase2_dynamic_loop(first_directive)
        final_data = self._phase3_orchestrator_close()

        score = final_data.get("preference_score")
        label = final_data.get("preference_label")
        reason = final_data.get("reason")
        elapsed = round(time.time() - start, 1)

        seq_html = "".join(
            f'<span style="background:{AGENT_ROLES[e["role"]]["color"]};color:white;'
            f'padding:2px 8px;border-radius:10px;margin:2px;font-size:12px;display:inline-block;">'
            f'T{e["turn"]} {e["role"]}</span>'
            for e in self.activation_log
        )

        score_val = score or 0
        sc = "#E74C3C" if score_val < 40 else ("#F39C12" if score_val < 60 else "#27AE60")
        sd = str(score) if score is not None else "파싱 실패"

        display(HTML(
            f'<div style="border:2px solid {sc};border-radius:10px;padding:20px;margin:20px 0;">'
            f'<h3 style="color:{sc};margin:0 0 12px;">📊 최종 선호도 출력</h3>'
            f'<table style="width:100%;font-size:14px;border-collapse:collapse;">'
            f'<tr style="border-bottom:1px solid #eee;"><td style="padding:6px;width:160px;"><b>선호도 점수</b></td>'
            f'<td><span style="font-size:28px;color:{sc};font-weight:bold;">{sd}</span> / 100</td></tr>'
            f'<tr style="border-bottom:1px solid #eee;"><td style="padding:6px;"><b>선호도 레이블</b></td>'
            f'<td style="padding:6px;">{label or "파싱 실패"}</td></tr>'
            f'<tr style="border-bottom:1px solid #eee;"><td style="padding:6px;"><b>사용 턴</b></td>'
            f'<td style="padding:6px;">{self.turn_count} / {MAX_DEBATE_TURNS}</td></tr>'
            f'<tr style="border-bottom:1px solid #eee;"><td style="padding:6px;"><b>소요 시간</b></td>'
            f'<td style="padding:6px;">{elapsed}초</td></tr>'
            f'<tr><td style="padding:6px;"><b>활성화 순서</b></td>'
            f'<td style="padding:6px;">{seq_html}</td></tr>'
            f'</table>'
            f'<div style="margin-top:12px;padding:10px;background:#f0f0f0;border-radius:4px;'
            f'font-size:13px;line-height:1.7;">'
            f'<b>이유:</b> {reason or "이유 없음"}</div>'
            f'</div>'
        ))

        return {
            "session_id":          self.session_id,
            "model":               self.model_name,
            "persona_summary":     self.persona[:200],
            "task":                self.task,
            "total_turns":         self.turn_count,
            "activation_sequence": [e["role"] for e in self.activation_log],
            "preference_score":    score,
            "preference_label":    label,
            "reason":              reason,
            "elapsed_sec":         elapsed,
            "debate_history":      self.debate_history,
        }

print("✅ Simulation engine ready (Structured Output 기반 — 파싱 오류 원천 차단)")

## ▶️ Run Simulation

In [ ]:
sim = PreferenceSimulationSystem(
    model_name=SELECTED_MODEL,
    persona=PERSONA,
    task=TASK
)

result = sim.run()

## 💾 Export Results (CSV & Excel)

In [ ]:
def export_results(result):
    sid = result["session_id"]

    # ── 1. Summary row ───────────────────────────────────────────
    summary_df = pd.DataFrame([{
        "session_id":           result["session_id"],
        "timestamp":            result["session_id"],
        "model":                result["model"],
        "task":                 result["task"],
        "persona_summary":      result["persona_summary"],
        "activation_sequence":  ' → '.join(result.get("activation_sequence", [])),
        "preference_score":     result["preference_score"],
        "preference_label":     result["preference_label"],
        "reason":               result["reason"],
        "total_turns":          result["total_turns"],
        "elapsed_sec":          result["elapsed_sec"],
    }])

    # ── 2. Debate log ────────────────────────────────────────────
    debate_rows = []
    for i, entry in enumerate(result["debate_history"]):
        debate_rows.append({
            "session_id": sid,
            "turn":       i + 1,
            "agent":      entry["agent"],
            "content":    entry["content"],
        })
    debate_df = pd.DataFrame(debate_rows)

    # ── CSV export ───────────────────────────────────────────────
    summary_csv = f"summary_{sid}.csv"
    debate_csv  = f"debate_log_{sid}.csv"
    summary_df.to_csv(summary_csv, index=False, encoding="utf-8-sig")
    debate_df.to_csv(debate_csv, index=False, encoding="utf-8-sig")

    # ── Excel export (multi-sheet) ───────────────────────────────
    excel_file = f"preference_result_{sid}.xlsx"
    with pd.ExcelWriter(excel_file, engine="openpyxl") as writer:
        summary_df.to_excel(writer, sheet_name="Summary", index=False)
        debate_df.to_excel(writer, sheet_name="Debate Log", index=False)

        # Style summary sheet
        wb = writer.book
        ws = writer.sheets["Summary"]
        from openpyxl.styles import PatternFill, Font, Alignment
        header_fill = PatternFill(start_color="2C3E50", end_color="2C3E50", fill_type="solid")
        header_font = Font(color="FFFFFF", bold=True)
        for cell in ws[1]:
            cell.fill = header_fill
            cell.font = header_font
            cell.alignment = Alignment(horizontal="center")

        # Auto-width
        for ws_name in ["Summary", "Debate Log"]:
            ws_obj = writer.sheets[ws_name]
            for col in ws_obj.columns:
                max_len = max(len(str(c.value or "")) for c in col)
                ws_obj.column_dimensions[col[0].column_letter].width = min(max_len + 4, 60)

    print("\n📁 Export complete:")
    print(f"  📄 {summary_csv}")
    print(f"  📄 {debate_csv}")
    print(f"  📊 {excel_file}")

    display(HTML(f"""
    <div style="background:#eafaf1; border: 1px solid #27AE60; border-radius:8px; padding:15px; margin:10px 0;">
      <h4 style="color:#27AE60; margin:0 0 8px;">✅ Files Exported</h4>
      <ul style="margin:0; font-family:monospace;">
        <li>📄 {summary_csv} (요약)</li>
        <li>📄 {debate_csv} (토론 로그)</li>
        <li>📊 {excel_file} (전체 - 멀티시트)</li>
      </ul>
      <p style="margin:8px 0 0; font-size:12px; color:#555;">Colab 좌측 파일 탭에서 다운로드하세요.</p>
    </div>"""))

    return summary_df, debate_df

summary_df, debate_df = export_results(result)
print("\n=== Summary ===")
display(summary_df.T)

## 🔁 Run Another Simulation (Optional)
다른 페르소나나 태스크로 추가 시뮬레이션을 실행하려면 아래 셀을 사용하세요.

In [ ]:
# ──────────────────────────────────────────────
# 추가 시뮬레이션 — 원하는 페르소나/태스크로 수정
# ──────────────────────────────────────────────

NEW_PERSONA = """
<static attributes>
Name: Lee Junhyeok
Age: 35 (Millennial)
Gender: Male
Personality (OCEAN): Openness(high) / Conscientiousness(high) / Extraversion(low) / Agreeableness(medium) / Neuroticism(medium)
Values (top 3): Universalism / Achievement / Self-Direction
Tech familiarity: Medium / Purchases after thorough evaluation / Perceives wearables as long-term investments
Sensory sensitivity: Average sensitivity to noise / Sensitive to fit and comfort of devices
Meta-traits: Stability / Cognitive Depth / Value Coherence

<dynamic context>
Physical surroundings: Home office while working remotely
Social context: Alone at home, working
Temporal perspective: Relaxed afternoon
Antecedent state: High concentration, relaxed mood, curiosity toward new technology
"""

NEW_TASK = """AI Buds Real-Time Health Monitoring: A feature that measures heart rate, stress index, and ear temperature in real time, sending an alert to the user's smartphone when anomalies are detected."""

run_new = input("추가 시뮬레이션을 실행하시겠습니까? (y/n, 기본값: n): ").strip().lower()
if run_new == 'y':
    sim2 = PreferenceSimulationSystem(
        model_name=SELECTED_MODEL,
        persona=NEW_PERSONA,
        task=NEW_TASK
    )
    result2 = sim2.run()
    export_results(result2)
else:
    print("추가 시뮬레이션 건너뜀")

## 📊 Batch Run (복수 태스크 평가)
동일 페르소나에 여러 기능을 한 번에 평가하여 비교할 수 있습니다.

In [ ]:
# 동일 페르소나 × 복수 태스크 배치 평가
BATCH_TASKS = [
    "AI Buds Spatiotemporal Adaptive Noise Control: Uses GPS and microphones to analyze the user's location and environment in real time, automatically optimizing noise canceling mode and intensity to match the current situation.",
    "AI Buds Emotion-Aware Music Recommendation: Analyzes the user's heart rate and speech tone to automatically recommend music suited to their current emotional state.",
    "AI Buds Call Quality Enhancement: Uses AI to filter surrounding noise in real time, ensuring only a clean voice is delivered to the caller at all times.",
]

run_batch = input(f"배치 평가({len(BATCH_TASKS)}개 태스크)를 실행하시겠습니까? (y/n, 기본값: n): ").strip().lower()

if run_batch == 'y':
    all_results = []
    for i, task_item in enumerate(BATCH_TASKS):
        display(HTML(f"<h3>🔄 Batch [{i+1}/{len(BATCH_TASKS)}]</h3>"))
        sim_b = PreferenceSimulationSystem(
            model_name=SELECTED_MODEL,
            persona=PERSONA,
            task=task_item
        )
        res = sim_b.run()
        all_results.append(res)
        time.sleep(1)

    # Comparison table
    comparison_rows = []
    for res in all_results:
        comparison_rows.append({
            "task": res["task"][:80] + "...",
            "score": res["preference_score"],
            "label": res["preference_label"],
            "reason": (res["reason"] or "")[:120],
        })
    comp_df = pd.DataFrame(comparison_rows)

    batch_ts = datetime.now().strftime("%Y%m%d_%H%M%S")
    comp_df.to_csv(f"batch_comparison_{batch_ts}.csv", index=False, encoding="utf-8-sig")
    comp_df.to_excel(f"batch_comparison_{batch_ts}.xlsx", index=False)

    print("\n📊 배치 비교 결과:")
    display(comp_df)
else:
    print("배치 평가 건너뜀")